# Verificación del código frente al artículo

Este notebook comprueba, una por una, que el paquete `schelling_matricial`
implementa la dinámica descrita en `docs/Articulo_Matriz_Promedio/main.tex`
(modelo P, promediado). Cada sección termina con un `assert`: si algo no
coincide con el artículo, la celda falla con un mensaje. Si todo está bien,
imprime `OK`.

| Sección | Qué se comprueba | Artículo |
|---|---|---|
| 1 | Con $H^{Cl}$ la incomodidad es la clásica $d_a/n_a$ | §2.1 |
| 2 | Aislados: $I_a^H = \iota_i = \max_j H_{i,j}$ | §2 |
| 3 | Regla de movimiento, movimiento a movimiento | §2 |
| 4 | Invariancia ante $\widetilde H = \alpha H + \beta\mathbf 1$ | §3 |
| 5 | Incomodidad total: $\Delta I^{\widetilde H} = \alpha\,\Delta I^H$ y movimientos con $\Delta I^H > 0$ | §3.1 |
| 6 | Orientación de cada tipo (homofílico, neutral, heterofílico) | §4 |
| 7 | Reducción al modelo clásico y regímenes | §4, Teorema 1 |

Notación: el tipo $1$ del artículo es el tipo `1` del código y el tipo $-1$ es
el tipo `2`. Las incomodidades se calculan como fracciones exactas
(`Fraction`), así que las comparaciones son exactas.

Usa el kernel del entorno `.venv_schelling`.

In [ ]:
from fractions import Fraction as F

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from schelling_matricial import Config, Simulacion
from schelling_matricial.geometria import distancia_chebyshev

# Colores de la retícula: 0 = vacío (gris claro), 1 = tipo 1, 2 = tipo -1.
COLORES = ListedColormap(["#e8e8e8", "#1f77b4", "#ff7f0e"])


def vecinos_moore(reticula, f, c):
    # Tipos de las 8 casillas vecinas (Moore, radio 1) en el toro.
    L = reticula.shape[0]
    return [reticula[(f + df) % L, (c + dc) % L]
            for df in (-1, 0, 1) for dc in (-1, 0, 1) if (df, dc) != (0, 0)]

## 1. Retorno al modelo clásico (§2.1)

Con $H^{Cl} = \begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix}$ el artículo
afirma que $I_a^{H^{Cl}} = d_a / n_a$, la fracción de vecinos ocupados de
otro tipo. Contamos $d_a$ y $n_a$ a mano en la retícula y lo comparamos con
la incomodidad que calcula el paquete para cada agente.

In [ ]:
cfg = Config(L=20, poblaciones=(150, 150), H=[[0, 1], [1, 0]], tau=F(1, 2), seed=1)
sim = Simulacion(cfg)

diferencias = 0
for (f, c), tipo, I in zip(sim.posiciones, sim.tipos, sim.incomodidades()):
    ocupados = [t for t in vecinos_moore(sim.reticula, f, c) if t > 0]
    n_a = len(ocupados)
    d_a = sum(t != tipo for t in ocupados)
    esperado = F(d_a, n_a) if n_a > 0 else max(F(1), F(0))   # aislado: max_j H[i, j] = 1
    diferencias += I != esperado

assert diferencias == 0, f"{diferencias} agentes con incomodidad distinta de d/n"
print(f"OK: los {len(sim.tipos)} agentes tienen I = d_a / n_a")

## 2. Agentes aislados (§2)

Si un agente no tiene vecinos ocupados ($n_a = 0$), el artículo fija
$I_a^H = \iota_{t_a}$ con $\iota_i = \max_j H_{i,j}$: el aislamiento vale
como la peor vecindad posible. En el código es `iota="max"`, el valor por
defecto.

Usamos una matriz no trivial y una retícula hecha a mano con un agente de cada
tipo, lejos uno del otro, de modo que los dos están aislados.

In [ ]:
H = [[F(1, 5), F(9, 10)], [F(2, 5), F(1, 10)]]
iota_esperado = [max(fila) for fila in H]        # [9/10, 2/5]

cfg = Config(L=7, poblaciones=(1, 1), H=H, tau=F(1, 2), seed=0)
print("iota según el paquete:", cfg.parametros()["iota"])

reticula = np.zeros((7, 7), dtype=int)
reticula[0, 0] = 1      # agente de tipo 1
reticula[3, 3] = 2      # agente de tipo -1, a distancia 3: no son vecinos
sim = Simulacion(cfg, reticula=reticula)

I_1 = sim.incomodidad((0, 0))
I_2 = sim.incomodidad((3, 3))
print("incomodidad de los aislados:", I_1, I_2)

assert [I_1, I_2] == iota_esperado
print("OK: un agente aislado tiene I = max_j H[i, j]")

## 3. Regla de movimiento (§2)

El artículo dice que un agente $a$ en $x$:

1. se mueve solo si está incómodo, $I_x^H(a) > \tau$;
2. solo a un vacío $y$ que mejore estrictamente, $I_y^H(a) < I_x^H(a)$, sin
   contarse a sí mismo como vecino de $y$;
3. de entre esos vacíos, al más cercano (distancia de Chebyshev en el toro),
   con desempate al azar;
4. el destino no tiene que ser cómodo.

Para comprobarlo, `ejecutar(al_mover=...)` llama a una función tras cada
movimiento. En ella reconstruimos el estado justo **antes** del movimiento,
evaluamos todos los vacíos y comprobamos que el destino elegido cumple 1–3.
También contamos cuántos movimientos acaban en un destino incómodo (punto 4).

In [ ]:
cfg = Config(L=20, poblaciones=(170, 170), H=[[0, 1], [1, 0]], tau=F(1, 2), seed=7)
sim = Simulacion(cfg)
tau = F(1, 2)

fallos = []
movimientos = 0
destino_incomodo = 0


def comprobar(agente, tipo, x, y, I_x, I_y):
    global movimientos, destino_incomodo
    movimientos += 1
    # Estado antes del movimiento: el agente vuelve a x y y queda vacío.
    antes = sim.reticula.copy()
    antes[y] = 0
    antes[x] = tipo
    previa = Simulacion(cfg, reticula=antes)

    vacios = list(zip(*np.nonzero(antes == 0)))
    mejoran = [v for v in vacios if previa.incomodidad(v, tipo, excluir=x) < I_x]
    filas = np.array([v[0] for v in mejoran])
    cols = np.array([v[1] for v in mejoran])
    d_min = distancia_chebyshev(x[0], x[1], filas, cols, cfg.L).min()
    d_y = distancia_chebyshev(x[0], x[1], y[0], y[1], cfg.L)

    if not (I_x > tau):
        fallos.append(("se movió estando cómodo", x, y))
    if not (I_y < I_x and y in mejoran):
        fallos.append(("el destino no mejora", x, y))
    if d_y != d_min:
        fallos.append(("el destino no es el más cercano", x, y))
    destino_incomodo += I_y > tau


resultado = sim.ejecutar(al_mover=comprobar)
print(f"Movimientos comprobados: {movimientos}  |  parada: {resultado['motivo_parada']}"
      f" / {resultado['tipo_punto_fijo']}")
print(f"Movimientos a un destino incómodo (permitidos por la regla formal): {destino_incomodo}")

assert movimientos > 0 and not fallos, fallos[:5]
print("OK: todos los movimientos cumplen la regla del artículo")

## 4. Invariancia ante transformaciones afines positivas (§3)

Con $\widetilde H = \alpha H + \beta \mathbf 1$ ($\alpha > 0$) y
$\widetilde\tau = \alpha\tau + \beta$, el artículo afirma que las
trayectorias son idénticas movimiento a movimiento, también con agentes
aislados, porque $\widetilde\iota = \alpha\iota + \beta$.

Usamos baja densidad para que haya agentes y vacíos aislados, y comparamos la
retícula tras cada barrido. **No** se transforma `iota` a mano: `iota="max"`
lo hace solo.

In [ ]:
H = np.array([[F(1, 5), F(9, 10)], [F(2, 5), F(1, 10)]], dtype=object)
tau = F(1, 2)
alpha, beta = F(3), F(-2)

H_t = (alpha * H + beta).tolist()
tau_t = alpha * tau + beta


def trayectoria(cfg, barridos=40):
    sim = Simulacion(cfg)
    estados = [sim.reticula.copy()]
    for _ in range(barridos):
        movs = sim.barrido()
        estados.append(sim.reticula.copy())
        if movs == 0:
            break
    return estados


base = dict(L=20, poblaciones=(60, 60), seed=3)
t_original = trayectoria(Config(**base, H=H.tolist(), tau=tau))
t_transformada = trayectoria(Config(**base, H=H_t, tau=tau_t))

iguales = len(t_original) == len(t_transformada) and all(
    np.array_equal(a, b) for a, b in zip(t_original, t_transformada))
print(f"Barridos: {len(t_original) - 1}  |  H~ = {H_t}, tau~ = {tau_t}")
n_aislados = sum(1 for (f, c) in zip(*np.nonzero(t_original[0]))
                 if all(t == 0 for t in vecinos_moore(t_original[0], f, c)))
print(f"Agentes aislados en el estado inicial: {n_aislados}")

assert len(t_original) > 2 and iguales
print("OK: H y alpha·H + beta producen la misma trayectoria")

## 5. Incomodidad total (§3.1)

La incomodidad total es $I^H(\eta) = \sum_a I_a^H$. El artículo afirma:

- $\Delta I^{\widetilde H} = \alpha\,\Delta I^H$ entre dos configuraciones
  cualesquiera;
- una mejora individual **no** implica que $I^H$ baje (Remark de §3.1).

Calculamos $I^H$ exacta tras cada movimiento, con las dos matrices de la
sección anterior, y contamos los movimientos en los que $I^H$ sube.

In [ ]:
def incomodidad_total_por_movimiento(cfg):
    sim = Simulacion(cfg)
    totales = [sum(sim.incomodidades())]
    sim.ejecutar(al_mover=lambda *args: totales.append(sum(sim.incomodidades())))
    return totales


base = dict(L=30, poblaciones=(400, 400), seed=1)
I_orig = incomodidad_total_por_movimiento(Config(**base, H=H.tolist(), tau=tau))
I_trans = incomodidad_total_por_movimiento(Config(**base, H=H_t, tau=tau_t))

dI_orig = np.diff(np.array(I_orig, dtype=object))
dI_trans = np.diff(np.array(I_trans, dtype=object))
suben = int(sum(d > 0 for d in dI_orig))

print(f"Movimientos: {len(dI_orig)}  |  con Delta I^H > 0: {suben}")
print(f"I^H inicial = {float(I_orig[0]):.3f}, final = {float(I_orig[-1]):.3f}")

assert len(dI_orig) == len(dI_trans) and all(dt == alpha * d for d, dt in zip(dI_orig, dI_trans))
print("OK: Delta I~ = alpha · Delta I en cada movimiento")
assert suben > 0
print("OK: hay movimientos que mejoran al agente pero aumentan I^H")

fig, eje = plt.subplots(figsize=(7, 3.5))
eje.plot([float(x) for x in I_orig])
eje.set(xlabel="Movimiento", ylabel="$I^H$", title="Incomodidad total tras cada movimiento")
plt.show()

## 6. Orientación de cada tipo (§4)

Para cada tipo $i$, con $h_i = H_{i,i}$ (vecino igual) y $\bar h_i = H_{i,-i}$
(vecino distinto), el artículo define la pendiente $\Delta_i = \bar h_i - h_i$:
el tipo es homofílico si $\Delta_i > 0$, neutral si $\Delta_i = 0$ y
heterofílico si $\Delta_i < 0$. En el código, $\Delta_1$ = `H[0][1] - H[0][0]`
y $\Delta_{-1}$ = `H[1][0] - H[1][1]`.

Como cada tipo solo depende de su fila, simulamos las 9 combinaciones de
orientaciones (3 del tipo 1 × 3 del tipo −1). Como umbral de cada tipo usamos
el punto medio de su fila, $\tau_i = (\min_j H_{i,j} + \max_j H_{i,j}) / 2$,
que da $\theta_i = 1/2$ en todos los tipos no neutrales: así las combinaciones
son comparables.

Medimos la fracción media de vecinos del mismo tipo, `C_occ[i, i]`, en el
estado final (media de 5 réplicas). Al azar vale
$\approx (N_i - 1)/(N - 1) \approx 0{,}5$. Esperamos:

- tipo homofílico: claramente **por encima** de la referencia;
- tipo heterofílico: claramente **por debajo**;
- tipo neutral: no se comprueba; se mueve de forma pasiva, arrastrado por el
  otro tipo.

In [ ]:
def clase(x):
    return "homofílico" if x > 0 else "heterofílico" if x < 0 else "neutral"


N1 = N2 = 400
referencia = (N1 - 1) / (N1 + N2 - 1)
filas = []
finales = {}

for b in (1, -1, 0):
    for c, d in ((1, 0), (0, 0), (0, 1)):
        H_tab = [[0, b], [c, d]]
        tau_tab = [F(min(fila) + max(fila), 2) for fila in H_tab]
        mismo_tipo = []
        for seed in range(5):
            sim = Simulacion(Config(L=30, poblaciones=(N1, N2), H=H_tab, tau=tau_tab, seed=seed))
            r = sim.ejecutar()
            mismo_tipo.append(np.diag(r["C_occ"]))
        finales[str(H_tab)] = sim.reticula.copy()
        m = np.mean(mismo_tipo, axis=0)
        filas.append({"H": str(H_tab), "Delta_1": b, "Delta_-1": c - d,
                      "tipo 1": clase(b), "tipo -1": clase(c - d),
                      "C_11": round(m[0], 3), "C_22": round(m[1], 3)})

tabla = pd.DataFrame(filas)
print(f"Referencia al azar: {referencia:.3f}")
tabla

In [ ]:
margen = 0.02
errores = []
for _, fila in tabla.iterrows():
    for tipo, col in (("tipo 1", "C_11"), ("tipo -1", "C_22")):
        if fila[tipo] == "homofílico" and not fila[col] > referencia + margen:
            errores.append((fila["H"], tipo))
        if fila[tipo] == "heterofílico" and not fila[col] < referencia - margen:
            errores.append((fila["H"], tipo))

assert not errores, errores
print("OK: cada tipo se comporta según su orientación (sección 4)")

fig, ejes = plt.subplots(3, 3, figsize=(10, 10))
for eje, (nombre, reticula), (_, fila) in zip(ejes.flat, finales.items(), tabla.iterrows()):
    eje.imshow(reticula, cmap=COLORES, vmin=0, vmax=2, interpolation="nearest")
    eje.set_title(f"H = {nombre}\n1: {fila['tipo 1']}, -1: {fila['tipo -1']}", fontsize=9)
    eje.axis("off")
plt.suptitle("Estado final (última réplica) para cada combinación de orientaciones")
plt.tight_layout()
plt.show()

## 7. Reducción al modelo clásico y regímenes (§4, Teorema 1)

El Teorema 1 afirma que, con dos tipos, el modelo P con $(H, \tau)$ es
exactamente un Schelling clásico en el que cada tipo tiene una orientación
$s_i = \operatorname{sgn} \Delta_i$ y un umbral reducido $\theta_i$, y en el que
un agente aislado vale 1. Además, cada tipo está en un régimen: inerte
($\theta_i \ge 1$), activo ($0 \le \theta_i < 1$) o siempre incómodo
($\theta_i < 0$); un tipo neutral es inerte o está congelado.

`parametros_reducidos(cfg)` calcula $(s_i, \theta_i)$ y el régimen, y
`config_reducida(cfg)` construye la configuración clásica equivalente.
Comprobamos que las dos dan la misma trayectoria, barrido a barrido, con
densidad alta y con densidad baja (con agentes aislados).

In [ ]:
from schelling_matricial import config_reducida, parametros_reducidos


def mismas(t1, t2):
    return len(t1) == len(t2) and all(np.array_equal(a, b) for a, b in zip(t1, t2))


casos = {
    "ejemplo del documento": ([[F(2, 5), F(3, 5)], [F(3, 4), F(3, 10)]], F(513, 1000)),
    "homofílico / heterofílico": ([[F(2, 5), F(3, 5)], [F(3, 10), F(3, 4)]], F(513, 1000)),
    "homofílico / neutral congelado": ([[0, 1], [F(3, 5), F(3, 5)]], F(1, 2)),
    "siempre incómodo (sección 4)": ([[F(3, 5), F(4, 5)], [F(4, 5), F(3, 5)]], F(1, 2)),
}

filas = []
for nombre, (H_c, tau_c) in casos.items():
    identicas = 0
    for poblaciones in ((150, 150), (60, 60)):
        for seed in range(3):
            cfg_c = Config(L=20, poblaciones=poblaciones, H=H_c, tau=tau_c, seed=seed)
            identicas += mismas(trayectoria(cfg_c, 60), trayectoria(config_reducida(cfg_c), 60))
    p = parametros_reducidos(cfg_c)
    filas.append({"caso": nombre,
                  "s": [x["s"] for x in p],
                  "theta": [str(x["theta"]) for x in p],
                  "régimen": [x["regimen"] for x in p],
                  "idénticas": f"{identicas} / 6"})

reduccion = pd.DataFrame(filas)
assert all(f["idénticas"] == "6 / 6" for f in filas)
print("OK: el modelo P coincide con el clásico reducido en todos los casos")
reduccion

## Resumen

Si todas las celdas han impreso `OK`, el código implementa la dinámica del
artículo:

- incomodidad promediada sobre los vecinos ocupados y modelo clásico como caso
  particular;
- aislados con $\iota_i = \max_j H_{i,j}$;
- regla de movimiento: incómodo si $I > \tau$, mejora estricta, vacío más
  cercano con desempate al azar, sin exigir destino cómodo;
- invariancia afín, también para los aislados y para la incomodidad total;
- comportamiento homofílico / heterofílico según la sección 4;
- reducción exacta al modelo clásico con $(s_i, \theta_i)$ (Teorema 1) y
  regímenes de cada tipo.

Los parámetros (`L`, poblaciones, semillas) son pequeños para que el notebook
corra en pocos segundos; se pueden subir para análisis más finos.